In [2]:
from pypdf import PdfReader

# Read the PDF
reader = PdfReader("dummyData.pdf")
text = "\n".join(page.extract_text() or "" for page in reader.pages)

# Chunk the document
chunks = [chunk.strip() for chunk in text.split("\n\n") if chunk.strip()]

# If chunks are too few, split by sentence
if len(chunks) < 5:
    chunks = [s.strip() + "." for s in text.split(".") if s.strip()]

print(f"Created {len(chunks)} chunks")

chunks


Created 6 chunks


['Space Exploration Knowledge Base \x7f RAG Demo Dataset\nPage 1\n SPACE EXPLORATION\n KNOWLEDGE BASE\n A document collection for building and testing a Retrieval-Augmented Generation (RAG)\n application\nDataset theme: planets, the Moon, Mars missions, spacecraft, black holes, exoplanets, and life-support\nsystems.\nPurpose\nThis PDF is designed as a small, self-contained knowledge source for a demonstration project. It contains short,\nclearly separated sections with facts and explanations that can be split into chunks, embedded, stored in a vector\ndatabase, and retrieved in response to natural-language questions.\nHow to use it\nUse this file as the source document for your RAG pipeline. Ask questions that match a specific section, ask the\nsame question using different wording, and ask questions whose answers are not included to test whether your\napplication admits uncertainty.\nImportant note\nThis is an educational dataset, not an official mission manual. It summarizes general 

In [3]:
from sentence_transformers import SentenceTransformer

#load the sentence transformer model
model = SentenceTransformer('all-MiniLM-L6-v2')

#generate embeddings for all chunks
embeddings = model.encode(chunks, show_progress_bar=True)
embeddings, len(embeddings)

import faiss
import numpy as np
import pickle

#create faiss index
dimension = embeddings.shape[1] #384 for all-MiniLM-L6-v2
index = faiss.IndexFlatL2(dimension)

#add embeddings to the index
index.add(np.array(embeddings).astype('float32'))

print(f"FAISS index created with {index.ntotal} vectors")
print(f"Embeddings dimension: {dimension}")

#save faiss index to diskk
faiss.write_index(index, 'dummyData_index.faiss')

#save the chunks for later retrieval
with open('dummyDatachunks.pkl', 'wb') as f:
          pickle.dump(chunks, f)

print("Index and chunks saved successfuly")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

FAISS index created with 6 vectors
Embeddings dimension: 384
Index and chunks saved successfuly


In [6]:
from autogen import AssistantAgent, register_function
import os
import json
import httpx
from autogen import LLMConfig
from environment_variables import base_url, api_key, TEMPERATURE
from environment_variables import model as llm_model  # Rename to avoid conflict
from autogen import UserProxyAgent
import warnings
import numpy as np

warnings.filterwarnings('ignore')


# Define the retrieval tool function
def space_context(query: str, top_k: int = 2) -> str:
    """
    Retrieve relevant policy context from the FAISS index based on user query.
    """

    # Encode the query using the existing sentence transformer model
    query_embedding = model.encode([query])

    # Search the FAISS index
    k = min(top_k, index.ntotal)

    if k == 0:
        return "No information is available in the index."

    distances, indices = index.search(
        np.array(query_embedding).astype('float32'), k
    )

    # Retrieve the corresponding chunks
    context_parts = []

    for i, (idx, distance) in enumerate(zip(indices[0], distances[0])):
        if 0 <= idx < len(chunks):
            context_parts.append(
                f"[Section {i+1}, Distance: {distance:.4f}]\n{chunks[idx]}"
            )

    if not context_parts:
        return "No relevant information was found."

    return "\n\n".join(context_parts)


# Custom HTTP client
class MyHttpClient(httpx.Client):
    def __deepcopy__(self, memo):
        return self


# Create the assistant agent
assistant = AssistantAgent(
    name="SpaceAssistant",

    system_message="""You are a helpful space research assistant.
    You have access to a tool that retrieves space related information
    from the provided data.

    When answering user questions:
    1. Use the space_context tool to get relevant information.
    2. Provide clear, accurate answers based on the space context.
    3. If the information is not available, clearly state
       that you do not have the information.
    4. Be professional and helpful in your responses.
    
    When you have generated the response, provide it to user by appending "TERMINATE",

    """,

    llm_config=LLMConfig(
        {
            "api_type": "openai",
            "model": llm_model,
            "base_url": base_url,
            "api_key": api_key,
            "max_tokens": 2000,
            "temperature": TEMPERATURE,
            "cache_seed": None,
            "seed": 42,
            "http_client": httpx.Client(verify=False),
            "price": [0.00015, 0.0006],
        }
    )
)


# Create a user proxy agent to interact with the assistant
user_proxy = UserProxyAgent(
    name="User",
    human_input_mode="NEVER",
    max_consecutive_auto_reply=2,
    code_execution_config=False
)


# Sample prompt to test the agent
sample_prompt = "How scientist find a black hole?"


# Register the retrieval tool with the agent
register_function(
    space_context,
    caller=assistant,
    executor=user_proxy,
    name="space_context",
    description= "Retrieves relevant information from the dataset based on a search query"
)


# Initiate the conversation
result = user_proxy.initiate_chat(
    assistant,
    message=sample_prompt,
    max_turns=10
)


User (to SpaceAssistant):

How scientist find a black hole?

--------------------------------------------------------------------------------
SpaceAssistant (to User):

***** Suggested tool call (fc_df450292-1bd5-4116-9424-c79f5f5bfc3d): space_context *****
Arguments: 
{"query":"how scientists find black holes methods detection black holes","top_k":5}
****************************************************************************************

--------------------------------------------------------------------------------

>>>>>>>> EXECUTING FUNCTION space_context...
Call ID: fc_df450292-1bd5-4116-9424-c79f5f5bfc3d
Input arguments: {'query': 'how scientists find black holes methods detection black holes', 'top_k': 5}

>>>>>>>> EXECUTED FUNCTION space_context...
Call ID: fc_df450292-1bd5-4116-9424-c79f5f5bfc3d
Input arguments: {'query': 'how scientists find black holes methods detection black holes', 'top_k': 5}
Output:
[Section 1, Distance: 1.0982]
Space Exploration Knowledge Base  RAG D